# Lab 02: From notebook to a command you can run again

Automated Workflows in Satellite Image Processing · v1.0 · 5 October 2026

Your Lab 01 service discovers real Sentinel scenes and stores its progress. Today you turn it into a program you can run from a terminal whenever your laptop is on.

Use this notebook to predict behavior, run checks and explain results. Edit **student_tasks.py** in a text editor. Run the actual service from a terminal. The service never depends on a notebook kernel.

By the end you can run the same command several times over a week, inspect its history, and explain whether it succeeded even when it found zero new scenes.

Suggested lab time: 90 minutes, followed by four short runs over seven days. No GPU, cluster or continuously running laptop is needed.


## What comes from Lab 01?

`lab01_service.py` contains the working schema, database insertion, pagination, overlapping discovery, and safe one-JPEG downloader extracted from Lab 01 instructor v3. It has no notebook execution side effects.

`workflow.py` supplies the orchestration and two small additions: a run journal and a link between each query stream and its scenes. The link prevents selecting a preview from a different region when a database contains several streams.

`sentinel_cli.py` supplies the argument parser. Your three repairs go in `student_tasks.py`. The captured provider pages and real preview in `fixtures/` are reused from Lab 01 only for repeatable checks. Operational discovery always uses live HTTPS requests; it never substitutes recorded data after a network failure.

Do not edit the supplied service or add a timer loop.


## Setup

Extract the student ZIP. Open a terminal in the extracted folder containing `sentinel_cli.py`, using your existing Python environment. If you need the notebook interface:

```bash
python -m pip install -r requirements.txt
```

```bash
python -m jupyterlab Lab02_Sentinel_CLI_v1_20261005.ipynb
```

The command line service and checks themselves need no third party packages. Keep the folder together. If you launch Jupyter from somewhere else, set `PROJECT` below to this folder's absolute path.


In [1]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys

PROJECT = Path.cwd().resolve()
assert (PROJECT / 'sentinel_cli.py').is_file(), 'Set PROJECT to the extracted student folder'

def run_check(exercise=None):
    args = [sys.executable, str(PROJECT / 'check_lab02.py')]
    if exercise is not None:
        args += ['--exercise', str(exercise)]
    result = subprocess.run(args, cwd=PROJECT, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print('Exit code:', result.returncode)
    return result.returncode

print('Project:', PROJECT)


Project: C:\Users\Natalia\Automated_Workflows\Labs\Lab02


## Before editing: inspect the command interface (5 minutes)

Run these in your terminal:

```bash
python sentinel_cli.py --help
```

```bash
python sentinel_cli.py discover --help
```

Read `config/krakow_week.json`. Where is the region specified? What makes this a repeatable query? Why is there an initial start but no permanent end date?


There are deliberate bugs in the three task functions. The historical demonstration end means the starter is not ready for the weekly experiment.


**Where is the region specified?**
 > The specified region is the area over Kraków and its surroundings.

**What makes this a repeatable query?**
> This is a repeatable query, because it doesn't have a specified end date, so with every re-run it can acquire new scenes.

**Why is there an initial start but no permanent end date?**
> The initial_start sets the historical baseline for data acquisition, while the absence of a permanent end date enables continuous monitoring by fetching newly published scenes up to the present date.

## Exercise 1: One configuration, one persistent state (20 minutes)

Read `resolve_paths` in `student_tasks.py`. Imagine running the program first from its own folder and then from its parent. Predict which database each invocation opens.

Repair the function so relative database, preview and log paths are anchored to the configuration file's parent. Keep absolute paths absolute. Resolve paths without creating files or changing the process working directory.

This matters because the scheduler or terminal decides the working directory. A successful command can silently create a second, empty catalogue if path handling is wrong.

Run the check before and after the repair:


In [2]:
run_check(1)


PASS: 1

Exit code: 0


0

Write your explanation here:

**Why is anchoring these paths to the configuration file more reliable than anchoring them to the terminal folder? What would happen if you deliberately changed the database path?**
> The terminal's working directory is chosen by whoever launches the program, if relative paths are resolved against that folder, the same command opens a different database depending on where it was started, it does not fail, it silently creates a new catalogue. 
> Anchoring these paths to the configuration file's parent makes the configuration self-contained. This was verified by running it from Lab02 and from its parent folder Labs and both reported the same database path, same scenes and same watermark. 
>
> If the database path was changed deliberately, the next run would create a new, empty database at the new location. The old database would remain untouched but unused and the history of the scenes would change, because every scene since initial start would be treated as new one.

## Exercise 2: Discover through the current time (20 minutes)

Read `choose_until`. The prototype copied Lab 01's demonstration end. That was useful for a fixed experiment, but it cannot discover this week's acquisitions.

Repair it so `--until` takes priority when supplied; otherwise capture an aware UTC timestamp once per invocation. Normalize explicit timestamps to UTC and reject values without a timezone. The optional `clock` argument lets a check control time without changing the satellite records.

Do not set the search start to “now minus one day.” The supplied discovery service uses the saved watermark minus the overlap, bounded by the original initial start.


In [3]:
run_check(2)


PASS: 2

Exit code: 0


0

**Write your explanation here: if you run on Monday and next on Thursday, which part of the configuration and saved state determines the next search interval? Why must the upper bound remain fixed while following pages?**

>The next search interval is not based on the weekday. It is determined by the saved state and the configuration. The start is the stream's saved watermark (the fixed upper bound of the last successful discovery) minus overlap days. The end is the current UTC time, captured once when the command starts. So if I run on Monday and next on Thursday, Thursday's search covers everything from Monday's end minus 7 days up to Thursday. 

>The upper bound must stay fixed while following pages because pagination assumes that the query does not change between pages. If the end moved forward during the run, newly published items could shift page boundaries, so items could be skipped or returned twice. Also, the watermark saved after the run would not describe the interval that was actually searched. With a single fixed `until`, every page belongs to the same query, and on success the watermark is set to exactly that `until`.


## Exercise 3: Make the terminal tell the truth (20 minutes)

Read `execute_job`. Predict what a caller sees when the network request fails. Repair the function so success returns exit code 0 and the actual result; failure logs a useful error and returns exit code 1 with `status='failed'` and an error description.

A completed search with zero new scenes is a successful run. A connection error is a failed run. Do not infer success from a positive scene count.

The check also launches the actual CLI with a missing configuration file to verify that the exit code reaches the terminal.


In [4]:
run_check(3)


PASS: 3

Exit code: 0


0

After your repair, try this deliberate failure in the terminal:

```bash
python sentinel_cli.py discover --config config/missing.json
```

Inspect the exit code immediately afterward. In Git Bash or Bash:

```bash
echo $?
```

In PowerShell:

```powershell
$LASTEXITCODE
```

**Explain what a future scheduler could do with this code. A missing configuration cannot provide a log path, so this early error appears on the terminal; operational errors also go to the configured log.**

>After the repair, the missing configuration returns and exit code 1. A future scheduler cannot read my JSON summary, but it can read the exit code. Exit code 0 means the command completed, even with zero new scenes, so there is nothing to do until the next scheduled run. A nonzero code means the run failed. The scheduler can then retry later, mark the job as failed, keep the log, or alert someone, instead of reporting success. Before the repair, a failed network request and a successful run both produced exit code 0, so automation could not tell "nothing new" from "the request failed".


## Exercise 4: Verify repeated execution (15 minutes)

Run every check. In addition to your three repairs, these reuse actual captured Lab 01 records to verify interrupted discovery, recovery, repeat sightings, and one verified preview. Only clocks and transport failures are controlled.


In [5]:
run_check()


PASS: 1
PASS: 2
PASS: 3
PASS: service recovery
PASS: one real preview

Exit code: 0


0

Now run the live program. Run one command at a time; simultaneous writers are outside this lab's scope. The first command discovers metadata and requests at most one preview. Preview download is optional on subsequent runs.

```bash
python sentinel_cli.py discover --config config/krakow_week.json --preview
```

```bash
python sentinel_cli.py discover --config config/krakow_week.json
```

```bash
python sentinel_cli.py report --config config/krakow_week.json
```

To demonstrate stable paths, move to the parent of the extracted folder and invoke the same program/configuration with their paths. Adapt the folder name to your extraction:

```bash
python Lab02_Sentinel_CLI_v1_20261005/sentinel_cli.py report --config Lab02_Sentinel_CLI_v1_20261005/config/krakow_week.json
```

Compare the reported absolute database path with the earlier report. It must be identical.

Live availability is not guaranteed. If a request fails, keep the log, inspect the saved watermark and retry later. Do not treat an offline check as evidence of a successful live run.


## Inspect your actual state

This cell only reads the current project's database. It does not perform discovery. If you changed the configured database path, update `DB` accordingly.


In [6]:
DB = (PROJECT / 'state' / 'catalogue.sqlite').resolve()
if not DB.is_file():
    print('Run live discovery from the terminal first; no database is created here.')
else:
    with sqlite3.connect(f'{DB.as_uri()}?mode=ro', uri=True) as conn:
        conn.row_factory = sqlite3.Row
        runs = [dict(row) for row in conn.execute(
            'SELECT started_at, status, search_start, search_until, pages, sightings, new_scenes, latest_acquired, error FROM runs ORDER BY started_at, rowid')]
        print(json.dumps(runs, indent=2))
        print('Saved query watermarks:')
        for row in conn.execute('SELECT stream_key, watermark FROM streams'):
            print(dict(row))


[
  {
    "started_at": "2026-11-04T14:42:04.378121Z",
    "status": "succeeded",
    "search_start": "2026-10-21T00:00:00Z",
    "search_until": "2026-11-04T14:42:04Z",
    "pages": 2,
    "sightings": 14,
    "new_scenes": 14,
    "latest_acquired": "2026-11-04T09:46:42.658000Z",
    "error": null
  },
  {
    "started_at": "2026-11-04T14:44:19.173163Z",
    "status": "succeeded",
    "search_start": "2026-10-28T14:42:04Z",
    "search_until": "2026-11-04T14:44:19Z",
    "pages": 2,
    "sightings": 6,
    "new_scenes": 0,
    "latest_acquired": "2026-11-04T09:46:42.658000Z",
    "error": null
  },
  {
    "started_at": "2026-11-07T07:31:16.798367Z",
    "status": "succeeded",
    "search_start": "2026-10-28T14:44:19Z",
    "search_until": "2026-11-07T07:31:16Z",
    "pages": 2,
    "sightings": 8,
    "new_scenes": 2,
    "latest_acquired": "2026-11-06T09:46:52.956000Z",
    "error": null
  },
  {
    "started_at": "2026-11-09T08:49:03.754556Z",
    "status": "succeeded",
    "searc

Explain your two consecutive runs. Which number counts sightings, and which counts scene identities first associated with this stream? Why may the second run have many sightings and zero new scenes? If new imagery arrived between runs, explain the difference rather than requiring a zero count.

If discovery fails after page 1, committed scenes remain and the discovery watermark does not advance. If discovery completes but its optional preview fails, the command fails while the discovery watermark has already advanced. These outcomes are recorded separately in the run journal and download status. Retry the same preview command to recover.

>`sightings` counts every item returned by the provider during a run, including items that were already in the database. `new_scenes` counts scene identities associated with this query stream for the first time in that run. My first run (2026-11-04 14:42) searched from `initial_start` (2026-10-21) and returned 14 sightings, which were all 14 new scenes. The immediate rerun two minutes later searched from the saved watermark minus the 7-day overlap (2026-10-28 14:42) and returned 6 sightings but 0 new scenes. Those 6 scenes were in the overlap window and had already been stored by the first run, so they were revisited, not duplicated. Both runs had status `succeeded` and exit code 0. Zero new scenes is a valid, complete result. If new imagery had been published between the two runs, the rerun would have shown a nonzero new-scene count, which would also have been correct.

## The seven day experiment

Run discovery at least four times over seven days whenever your laptop is on and online. Keep the same region, initial start and database throughout. Include one immediate rerun and one gap of at least 48 hours. A short run needs no permanent background process.

```bash
python sentinel_cli.py discover --config config/krakow_week.json --preview
```

You may omit `--preview` for metadata-only runs. It always selects the newest available suitable JPEG in this stream, without a cloud filter; a very cloudy preview is still a valid acquisition. It may reuse the same verified JPEG on several days. A JPEG is not a scientific raster product or a quality assessment.

At the end, save a report:

```bash
python sentinel_cli.py report --config config/krakow_week.json > week_report.json
```

Complete this table from the actual run history:

| Run time (UTC) | Outcome | Search interval | Sightings | Newly discovered scenes | Newest acquisition | Observation |
|---|---|---|---|---|---|---|
| 2026-11-04 14:42 | succeeded | 2026-10-21 00:00 → 2026-11-04 14:42 | 14 | 14 | 2026-11-04 09:46 | First run from initial_start; preview S2C_34UCA_20261104 downloaded |
| 2026-11-04 14:44 | succeeded | 2026-10-28 14:42 → 2026-11-04 14:44 | 6 | 0 | 2026-11-04 09:46 | Immediate rerun; the 7-day overlap revisited known scenes only |
| 2026-11-07 07:31 | succeeded | 2026-10-28 14:44 → 2026-11-07 07:31 | 8 | 2 | 2026-11-06 09:46 | After a gap of more than 48 h; resumed from saved watermark, no days lost; new preview S2A_34UCA_20261106 |
| 2026-11-09 08:49 | succeeded | 2026-10-31 07:31 → 2026-11-09 08:49 | 6 | 2 | 2026-11-07 09:56 | Second gap of ~49 h; newest scene ~47 h old at discovery; new preview S2C_34UCA_20261107 |
| 2026-11-09 08:55 | succeeded | 2026-10-21 00:00 → 2026-11-09 08:55 | 18 | 0 | 2026-11-07 09:56 | Reconciliation with --lookback-days 30, clamped to initial_start; no late-published scenes found |

Discuss: 
**Did skipping days lose scenes within the covered search interval?** 
>No. The laptop did nothing between 2026-11-04 14:44 and 2026-11-07 07:31, and again until 2026-11-09 08:49. Each later run started from the saved watermark minus the 7-day overlap, so the whole gap was searched. The 30-day reconciliation run, which re-searched the full interval from `initial_start`, returned 18 sightings and 0 new scenes. This is consistent with the database already containing all 18 scenes the provider returns for this interval. Each acquisition appears as two tiles (34UCA and 34UDA), and there are 9 acquisitions on 8 dates between 2026-10-23 and 2026-11-07 (on 2026-10-30 both Sentinel-2A and Sentinel-2B passed over the area).

**How old was the newest acquisition?**
>About 5 h at the first run, about 22 h at the third run, and about 47 h at the fourth run and in the final report.

**Did a new discovery necessarily mean a recent acquisition?**
>No. The two scenes that were new in the fourth run had been acquired almost two days earlier. They were "new" only because the previous run happened before that acquisition.

**What evidence distinguishes “no new scenes” from “the request failed”?**
>The run status, the exit code and the watermark. The immediate rerun had `status: succeeded`, exit code 0, `error: null`, a nonzero number of sightings with 2 committed pages, and the watermark advanced to its `search_until`. A failed request would have `status: failed`, exit code 1, an error message in the run journal and in `events.log`, and the discovery watermark would not advance. Zero new scenes on its own says nothing about success. Only the recorded outcome does.

**Was the newest preview also the clearest?**
>In the end yes, but only by coincidence, and not at every run. The service picks the newest available JPEG regardless of cloud cover. At the first two runs the newest preview was far from clear.

This is polling, not an immediate notification service. Acquisition time is not catalogue publication time. Finite overlap can miss items published very late with older acquisition dates; a complete search also cannot prove that the provider has every acquisition. Try a wider lookback as a reconciliation exercise:

```bash
python sentinel_cli.py discover --config config/krakow_week.json --lookback-days 30
```

This cannot go before `initial_start`. Set the initial start once before your first operational run; do not move it forward every day. The shipped start is 21 September 2026, two weeks before this edition. For a future teaching date, update it before starting the experiment.


## Submission

Submit your repaired `student_tasks.py`, this notebook with checks and explanations, `week_report.json`, and your completed observation table. Include a short relevant log excerpt if a live run failed. Do not commit preview images, database files or the entire log to Git.

Your service is now ready to be run by another program. Containers and cluster scheduling can be added in later labs without moving the operational logic back into the notebook.

Provider documentation: [Element 84 Earth Search](https://github.com/Element84/earth-search). Lab 01 source: `Lab01_Sentinel_RealData_Repair_v3_20261005_Instructor.ipynb`. Recorded fixtures retain the original provider records and real JPEG.
